In [ ]:
# Step 5: Wetlands Insight Tool case-study assessment

In [ ]:
# ===============================================================
# Step-5 — Wetlands Insight Tool case-study workflow
# Example wetland: Sunshower Lagoon
# Outputs:
#   1) WIT time-series CSV
#   2) Annual normalized WIT plot
# ===============================================================

import datetime
import itertools
import sys

import datacube
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.dates import YearLocator, DateFormatter
from matplotlib.patches import Patch

sys.path.insert(1, "../Tools/")
import dea_tools.bandindices
import dea_tools.datahandling
import dea_tools.wetlands
from dea_tools.dask import create_local_dask_cluster
from dea_tools.spatial import xr_rasterize

# Create local dask cluster to improve data load time
client = create_local_dask_cluster(return_client=True)

dc = datacube.Datacube(app="DEA_Wetlands_Insight_Tool")

# -------------------
# CASE-STUDY INPUT
# -------------------
polygon_path = "/home/jovyan/DEA products paper/Data/wetland boundaries/Sunshower Lagoon.shp"
polygon_name = "Sunshower"

# -------------------
# LOAD WETLAND POLYGON
# -------------------
poly = gpd.read_file(polygon_path)
poly.geometry[0]

# Specifying coordinate reference system of the polygon
gpgon = datacube.utils.geometry.Geometry(poly.geometry[0], crs=poly.crs)

# Select time period. Consistent data is available from 01-09-1987.
time = ("1988-01-01", "2024-01-01")

# Define which spectral bands are being used in the analysis
bands = [
    f"nbart_{band}" for band in ("blue", "green", "red", "nir", "swir_1", "swir_2")
]

# -------------------
# LOAD LANDSAT ARD
# -------------------
# Load Landsat 5, 7 and 8 data. Not including Landsat 7 SLC-off period.
ds = dea_tools.datahandling.load_ard(
    dc,
    products=["ga_ls8c_ard_3", "ga_ls7e_ard_3", "ga_ls5t_ard_3"],
    ls7_slc_off=False,
    measurements=bands,
    geopolygon=gpgon,
    output_crs="EPSG:3577",
    resolution=(-30, 30),
    resampling={"fmask": "nearest", "*": "bilinear"},
    time=time,
    group_by="solar_day",
    dask_chunks={"time": 1, "x": 2048, "y": 2048},
)

# Load into memory using Dask
ds.load()

# -------------------
# LOAD WOfS AND FRACTIONAL COVER
# -------------------
ds_wo = dc.load(
    "ga_ls_wo_3",
    resampling="nearest",
    group_by="solar_day",
    like=ds,
    dask_chunks={"time": 1, "x": 2048, "y": 2048},
)

ds_fc = dc.load(
    "ga_ls_fc_3",
    resampling="nearest",
    group_by="solar_day",
    like=ds,
    dask_chunks={"time": 1, "x": 2048, "y": 2048},
)

# Load data into memory
ds_wo.load()
ds_fc.load()

# -------------------
# ALIGN TIMES ACROSS DATASETS
# -------------------
missing = set()
for t1, t2 in itertools.product(
    [ds_fc.time.values, ds_wo.time.values, ds.time.values], repeat=2
):
    missing_ = set(t1) - set(t2)
    missing |= missing_

ds_fc = ds_fc.sel(time=[t for t in ds_fc.time.values if t not in missing])
ds = ds.sel(time=[t for t in ds.time.values if t not in missing])
ds_wo = ds_wo.sel(time=[t for t in ds_wo.time.values if t not in missing])

# -------------------
# CALCULATE TCW
# -------------------
tcw = dea_tools.bandindices.calculate_indices(
    ds,
    index="TCW",
    collection="ga_ls_3",
    normalise=False,
    drop=True,
    inplace=False,
)

# -------------------
# PREPARE WIT COMPONENTS
# -------------------
bs = ds_fc.bs / 100
pv = ds_fc.pv / 100
npv = ds_fc.npv / 100

rast_names = ["pv", "npv", "bs", "wet", "water"]
output_rast = {n: xr.zeros_like(bs) for n in rast_names}

output_rast["bs"].values[:] = bs
output_rast["pv"].values[:] = pv
output_rast["npv"].values[:] = npv

# Rasterise the shapefile using pv as the xarray template
poly_raster = xr_rasterize(poly, pv) > 0

# Mask includes NoData, non-contiguous data, cloud shadow, cloud, and water.
mask = (ds_wo.water & 0b01100011) == 0
mask &= poly_raster

# Set open water where WOfS classifies water present
open_water = ds_wo.water & (1 << 7) > 0

# Set wet pixels where not masked and TCW > -350
wet = tcw.where(mask).TCW > -350

# -------------------
# ADD WET AND WATER TO OUTPUT RASTERS
# -------------------
output_rast["wet"].values[:] = wet.values.astype(float)
for name in rast_names[:3]:
    output_rast[name].values[wet.values] = 0

output_rast["water"].values[:] = open_water.values.astype(float)
for name in rast_names[:4]:
    output_rast[name].values[open_water.values] = 0

# -------------------
# BUILD WIT DATASET
# -------------------
ds_wit = xr.Dataset(output_rast).where(mask)

# Calculate percentage missing
pc_missing = (~mask).where(poly_raster).mean(dim=["x", "y"])

ds_wit = ds_wit.where(pc_missing < 0.1)

# -------------------
# CONVERT TO DATAFRAME
# -------------------
polygon_base_df = pd.DataFrame()
polygon_base_df["date"] = ds_wit.time.values

for band in rast_names:
    polygon_base_df[band] = ds_wit[band].mean(dim=["x", "y"])

polygon_base_df = dea_tools.wetlands.normalise_wit(polygon_base_df)

png_name = polygon_name  # file will be png_name.png

dea_tools.wetlands.display_wit_stack_with_df(
    polygon_base_df,
    polygon_name,
    png_name,
    x_axis_labels="years"
)

output_df = polygon_base_df.set_index("date")
if "index" in output_df.columns:
    output_df = output_df.drop("index", axis=1)

# Write WIT dataframe to csv
output_df.to_csv(f"{polygon_name}.csv", index=True)

# -------------------
# PLOT YEARLY NORMALIZED WIT STACK
# -------------------
df = pd.read_csv(
    f"/home/jovyan/DEA products paper/Code/{polygon_name}.csv",
    parse_dates=["date"]
).sort_values("date")

stack_cols = ["water", "wet", "pv", "npv", "bs"]

colors = {
    "bs": "#8B1A1A",      # dark reddish brown for bare soil
    "npv": "#FFD700",     # yellow for dry vegetation
    "pv": "#006400",      # dark green for green vegetation
    "wet": "#00FFFF",     # cyan for wet
    "water": "#0000FF",   # blue for open water
}

# Row-normalize so each date sums to 1.0
X = df[stack_cols].copy()
row_sum = X.sum(axis=1).replace(0, np.nan)
X_norm = (X.T / row_sum).T.fillna(0)

df_norm = df.copy()
df_norm[stack_cols] = X_norm

# Create figure and axis
fig, ax = plt.subplots(figsize=(18, 4))
ax.set_xlabel("")
ax.margins(x=0)
ax.set_xlim(df_norm["date"].min(), df_norm["date"].max())
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.yaxis.set_ticks_position("left")
ax.xaxis.set_ticks_position("bottom")

# Main stackplot
ax.stackplot(
    df_norm["date"],
    *[df_norm[c] * 100 for c in stack_cols],
    colors=[colors[c] for c in stack_cols],
    labels=["open water", "wet", "green vegetation", "dry vegetation", "bare soil"],
    linewidth=0,
    antialiased=True,
)

# Yearly ticks with horizontal labels
ax.xaxis.set_major_locator(YearLocator(1))
ax.xaxis.set_major_formatter(DateFormatter("%Y"))
plt.xticks(rotation=0)
ax.tick_params(axis="x", labelsize=8, length=3, pad=2)

# Legend outside
legend_handles = [
    Patch(facecolor=colors["water"], label="open water"),
    Patch(facecolor=colors["wet"], label="wet"),
    Patch(facecolor=colors["pv"], label="green vegetation"),
    Patch(facecolor=colors["npv"], label="dry vegetation"),
    Patch(facecolor=colors["bs"], label="bare soil"),
]

ax.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=5,
    frameon=False,
    handlelength=2,
    handleheight=1.2,
    handletextpad=0.5,
    columnspacing=1.0,
)

# Labels and limits
ax.set_ylabel("Percentage of area (%)")
ax.set_ylim(0, 100)
ax.set_xlim(df_norm["date"].min(), df_norm["date"].max())
ax.margins(x=0)

# Clean look
for s in ("top", "right"):
    ax.spines[s].set_visible(False)

plt.tight_layout()
plt.savefig(f"{polygon_name}_WIT_yearly_normalized.png", dpi=300, bbox_inches="tight")
plt.show()